In [ ]:
# UCBT modeling 

import os
import json
import warnings
import joblib
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from pandas.api.types import is_numeric_dtype
from IPython.display import display

from sklearn.model_selection import (
    GroupShuffleSplit, StratifiedKFold, train_test_split, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline as SkPipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, roc_curve
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# SHAP 
try:
    import shap
    HAS_SHAP = True
    warnings.filterwarnings("ignore", category=UserWarning, module="shap")
except Exception:
    HAS_SHAP = False
    print("[WARN] SHAP unavailable; skipping explainability steps.")


# Config & paths
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE = Path("/Users/amanda/Desktop/UCBT")

# Choose which dataset to run:
# Uncoment ONE of the following lines per run
# Synthetic dataset (used for Approach 2)
DATA_PATH = BASE / "ucbt_dataset_synthetic_best.csv"   # synthetic
# Real dataset (used for Approach 1)
#DATA_PATH = BASE / "ucbt_dataset.csv" # real 
OUTDIR_NAME = DATA_PATH.stem.replace("ucbt_dataset_", "") 
OUTDIR = BASE / f"models_output_{OUTDIR_NAME}"
(OUTDIR / "metrics").mkdir(parents=True, exist_ok=True)
(OUTDIR / "models").mkdir(parents=True, exist_ok=True)
(OUTDIR / "shap").mkdir(parents=True, exist_ok=True)
(OUTDIR / "figs").mkdir(parents=True, exist_ok=True)
print("Using dataset:", DATA_PATH)
print("Using folder:", OUTDIR)


# Features/outcomes

MODEL_FEATURES = [
    "Recipient_Age", "Recipient_Sex", "Cord_Blood_Units", "Remission_Status",
    "CD34_num", "TNC_num", "hla_best", "hla_worst", "hla_any6", "hla_double",
    "HLAxCD34", "HLAxTNC", "Age_x_CD34", "Age_x_TNC", "log_CD34_num", "log_TNC_num",
    "Age2", "hla_mean", "hla_range", "CD34_adequate", "TNC_adequate", "Adequacy_Both",
    "Regimen_MA", "Age_x_RegimenMA", "CD34_num_q_by_DzHLA", "TNC_num_q_by_DzHLA"
]
BASE_CATS = ["Ethnicity", "Race", "Disease_Type", "Conditioning_Regimen", "HLA_Match_Level", "AgeBin"]
OUTCOMES = ["Neutrophil_Engraftment", "Platelet_Engraftment", "Chronic_GVHD", "1_Year_Survival"]

# OneHotEncoder config 
cat_encoder_kwargs = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    cat_encoder_kwargs["sparse_output"] = False
else:
    cat_encoder_kwargs["sparse"] = False


# Helpers
def best_threshold_for_roc_auc(y_true, proba, lo=0.20, hi=0.80, steps=61):
    grid = np.linspace(lo, hi, steps)
    aucs = [roc_auc_score(y_true, (proba >= t).astype(int)) for t in grid]
    return float(grid[np.argmax(aucs)])

class AlignTwoCols:
    """Rank-align CD34_num and TNC_num."""
    def __init__(self, cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99):
        self.cols = list(cols)
        self.q_lo = q_lo
        self.q_hi = q_hi
        self.params_ = {}

    def fit(self, df):
        self.params_.clear()
        for c in self.cols:
            if c not in df.columns:
                continue
            x = pd.to_numeric(df[c], errors="coerce").dropna().values
            if x.size == 0:
                continue
            ql, qh = np.quantile(x, [self.q_lo, self.q_hi])
            xs = np.sort(np.clip(x, ql, qh))
            self.params_[c] = {"ql": float(ql), "qh": float(qh), "sorted": xs}
        return self

    def transform(self, df):
        out = df.copy()
        for c, p in self.params_.items():
            if c not in out.columns:
                continue
            x = pd.to_numeric(out[c], errors="coerce").astype(float).clip(p["ql"], p["qh"])
            xs = p["sorted"]
            if xs.size <= 1:
                out[f"{c}__aligned"] = x
            else:
                ranks = np.searchsorted(xs, x, side="right") / xs.size
                out[f"{c}__aligned"] = ranks
        return out

    def fit_transform(self, df):
        return self.fit(df).transform(df)

def build_pipeline(preprocessor, model, use_smote: bool, model_name: str):
    """
    SMOTE is applied only for non-XGB models AND only when requested (use_smote).
    Preprocessor is a ColumnTransformer that returns a dense matrix,
    so SMOTE can consume it safely.
    """
    steps = [("preprocess", preprocessor)]
    if model_name != "xgb" and use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy="auto")))
    steps.append(("clf", model))
    return ImbPipeline(steps)

def make_model(model_name, outcome):
    if model_name == "xgb":
        return XGBClassifier(
            n_estimators=1200, max_depth=4, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
            reg_lambda=1.0, reg_alpha=0.0, n_jobs=-1, eval_metric="logloss",
            tree_method="hist", random_state=RANDOM_STATE
        )
    elif model_name == "gbm":
        return HistGradientBoostingClassifier(
            max_depth=4, learning_rate=0.05, max_iter=800,
            early_stopping=True, validation_fraction=0.15,
            random_state=RANDOM_STATE
        )
    elif model_name == "rf":
        return RandomForestClassifier(
            n_estimators=1200 if outcome in ("Neutrophil_Engraftment", "Platelet_Engraftment") else 1000,
            max_depth=16 if outcome in ("Neutrophil_Engraftment", "Platelet_Engraftment") else None,
            min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
        )
    elif model_name == "svm":
        return SVC(kernel="rbf", C=1.0, probability=True, random_state=RANDOM_STATE)
    raise ValueError(f"Unsupported model: {model_name}")

def plot_roc(pipe, X, y, title, path):
    prob = pipe.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, prob)
    auc = roc_auc_score(y, prob)
    plt.figure(figsize=(5.5, 4.5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(title); plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=220, bbox_inches="tight"); plt.close()


# Load & prepare
df = pd.read_csv(DATA_PATH)
drop_cols = [c for c in df.columns if c.endswith("_pre")]
df = df.drop(columns=drop_cols, errors="ignore")
feature_pool = [c for c in (MODEL_FEATURES + BASE_CATS) if c in df.columns]

# Train/Eval loop
all_rows = []
HOLDOUTS = {}
models_to_run = ["xgb", "gbm", "rf", "svm"]

for outcome in OUTCOMES:
    if outcome not in df.columns:
        print(f"Skipping {outcome}: column missing in data")
        continue

    y = df[outcome]
    if y.notna().sum() < 100 or y.nunique() < 2:
        print(f"Skipping {outcome}: insufficient data or single class")
        continue

    X = df[feature_pool].copy()
    y = y.astype(int)

    # Try both 70/30 and 80/20 splits
    for split_name, test_size in [("70_30", 0.30), ("80_20", 0.20)]:
        # Grouped split if Study_ID exists
        if "Study_ID" in df.columns:
            gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
            tr_pos, ho_pos = next(gss.split(X, y, groups=df["Study_ID"]))
            tr_idx, ho_idx = X.index[tr_pos], X.index[ho_pos]
        else:
            tr_idx, ho_idx = train_test_split(
                X.index, test_size=test_size, random_state=RANDOM_STATE, stratify=y
            )
        HOLDOUTS[f"{outcome}_{split_name}"] = (tr_idx, ho_idx)

        X_tr, X_te = X.loc[tr_idx].copy(), X.loc[ho_idx].copy()
        y_tr, y_te = y.loc[tr_idx].copy(), y.loc[ho_idx].copy()

        # Robust align CD34/TNC
        aligner = AlignTwoCols(cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99)
        X_tr = aligner.fit_transform(X_tr)
        X_te = aligner.transform(X_te)

        # Build column lists (after aligner adds *_aligned)
        num_cols = [c for c in X_tr.columns if is_numeric_dtype(X_tr[c])]
        cat_cols = [c for c in X_tr.columns if not is_numeric_dtype(X_tr[c])]

        # impute -> scale for numeric; impute -> OHE for categorical
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", SkPipeline([
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", MinMaxScaler())
                ]), num_cols),
                ("cat", SkPipeline([
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(**cat_encoder_kwargs))
                ]), cat_cols),
            ],
            remainder="drop",
            verbose_feature_names_out=False,
        )

        # Decide if SMOTE is needed (class imbalance on y_tr)
        cnt = Counter(y_tr)
        mn, mx = min(cnt.values()), max(cnt.values())
        ratio = mn / mx if mx else 0.0
        USE_SMOTE = (ratio < 0.80 and mn >= 6)  # avoid pathological resampling

        for model_name in models_to_run:
            model = make_model(model_name, outcome)
            pipe = build_pipeline(preprocessor, model, use_smote=USE_SMOTE, model_name=model_name)

            # 3-fold CV
            cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
            cv_scores = cross_validate(
                pipe, X_tr, y_tr, cv=cv,
                scoring=['roc_auc', 'accuracy', 'precision', 'recall'],
                return_train_score=False, n_jobs=-1
            )

            # Threshold tuning on a small validation slice from train
            tr_i, va_i = train_test_split(
                X_tr.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y_tr
            )
            pipe.fit(X_tr.loc[tr_i], y_tr.loc[tr_i])
            val_proba = pipe.predict_proba(X_tr.loc[va_i])[:, 1]
            t_star = best_threshold_for_roc_auc(y_tr.loc[va_i], val_proba)

            # Full training and test evaluation
            pipe.fit(X_tr, y_tr)
            proba_te = pipe.predict_proba(X_te)[:, 1]
            pred_te = (proba_te >= t_star).astype(int)

            auc_te = roc_auc_score(y_te, proba_te)
            acc_te = accuracy_score(y_te, pred_te)
            prec_te = precision_score(y_te, pred_te, zero_division=0)
            rec_te = recall_score(y_te, pred_te, zero_division=0)

            # Save model+threshold for 70/30 split (the primary one)
            if split_name == "70_30":
                model_path = OUTDIR / "models" / f"{model_name}_{outcome}.joblib"
                joblib.dump(pipe, model_path)
                threshold_path = OUTDIR / "models" / f"{model_name}_{outcome}_threshold.json"
                with open(threshold_path, "w") as f:
                    json.dump({"threshold": float(t_star)}, f)
                print(f"Saved model for {model_name} - {outcome} -> {model_path}")
                print(f"Saved threshold for {model_name} - {outcome} -> {threshold_path}")

            smote_applied = (model_name != "xgb" and USE_SMOTE)
            row = {
                "outcome": outcome,
                "split": split_name,
                "model": model_name,
                "cv_auc_mean": float(cv_scores['test_roc_auc'].mean()),
                "cv_auc_std": float(cv_scores['test_roc_auc'].std()),
                "cv_acc_mean": float(cv_scores['test_accuracy'].mean()),
                "test_auc": auc_te,
                "test_acc": acc_te,
                "test_prec": prec_te,
                "test_rec": rec_te,
                "used_smote": smote_applied,
                "auc_pass": auc_te >= 0.80
            }
            all_rows.append(row)

            print(f"[Split {split_name}] {model_name} - {outcome}: "
                  f"test_auc={auc_te:.3f}, cv_auc_mean={row['cv_auc_mean']:.3f}, SMOTE={smote_applied}")


# Save CV summary
cv_table = pd.DataFrame(all_rows).sort_values(
    ["outcome", "split", "model", "test_auc"],
    ascending=[True, True, True, False]
)
cv_table['auc_flag'] = np.where(cv_table['test_auc'] < 0.80, "FAIL", "PASS")
print("Validation Summary (3-fold CV, 70/30 and 80/20 splits):")
display(cv_table)

cv_path = OUTDIR / "metrics" / "cv_summary.csv"
cv_table.to_csv(cv_path, index=False)
print(f"Saved CV summary -> {cv_path}")

# Best models (70/30)
best_models = cv_table[cv_table['split'] == "70_30"].groupby('outcome').apply(
    lambda x: x.loc[x['test_auc'].idxmax()]
).reset_index(drop=True)

print("Best Models per Outcome (70/30 split):")
display(best_models[['outcome', 'model', 'test_auc', 'auc_flag', 'used_smote']])
best_models.to_csv(OUTDIR / "metrics" / "best_models.csv", index=False)
print(f"Saved best models -> {OUTDIR / 'metrics' / 'best_models.csv'}")

# SHAP (tree models)
if HAS_SHAP:
    for outcome in OUTCOMES:
        if outcome not in df.columns:
            continue
        y = df[outcome]
        if y.notna().sum() < 100 or y.nunique() < 2:
            continue

        X = df[feature_pool].copy()
        y = y.astype(int)

        for model_name in ["xgb", "gbm", "rf"]:  # SVM not supported by TreeExplainer
            model_path = OUTDIR / "models" / f"{model_name}_{outcome}_{OUTDIR_NAME}.joblib"
            if not model_path.exists():
                continue

            pipe = joblib.load(model_path)

            # Align then transform with the stored preprocessor
            aligner = AlignTwoCols(cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99)
            X = aligner.fit_transform(X)

            # Transform via preprocessor
            X_trans = pipe.named_steps["preprocess"].transform(X)
            X_dense = X_trans.toarray() if hasattr(X_trans, "toarray") else np.asarray(X_trans)

            try:
                feat_names = pipe.named_steps["preprocess"].get_feature_names_out()
            except Exception:
                # Fallback feature names
                n_cols = X_dense.shape[1]
                feat_names = [f"f{i}" for i in range(n_cols)]

            # Subsample for SHAP speed
            sample = min(2000, X_dense.shape[0])
            idx = np.random.choice(np.arange(X_dense.shape[0]), size=sample, replace=False)
            X_sample = X_dense[idx]

            try:
                explainer = shap.TreeExplainer(pipe.named_steps["clf"])
                shap_values = explainer.shap_values(X_sample)
                sv = shap_values[1] if isinstance(shap_values, list) and len(shap_values) > 1 else shap_values

                plt.figure()
                shap.summary_plot(sv, X_sample, feature_names=feat_names, show=False, max_display=20)
                out_png = OUTDIR / "shap" / f"shap_summary_{model_name}_{outcome}.png"
                plt.tight_layout(); plt.savefig(out_png, dpi=220, bbox_inches="tight"); plt.close()
                print(f"Saved SHAP summary -> {out_png}")

                plt.figure()
                shap.summary_plot(sv, X_sample, feature_names=feat_names, plot_type="bar", show=False, max_display=20)
                out_png2 = OUTDIR / "shap" / f"shap_bar_{model_name}_{outcome}.png"
                plt.tight_layout(); plt.savefig(out_png2, dpi=220, bbox_inches="tight"); plt.close()
                print(f"Saved SHAP bar -> {out_png2}")

                vals = np.abs(sv).mean(axis=0)
                top_k = min(25, len(vals))
                top_idx = np.argsort(-vals)[:top_k]
                top_df = pd.DataFrame({"feature": np.array(feat_names)[top_idx], "mean_abs_shap": vals[top_idx]})
                top_csv = OUTDIR / "shap" / f"top_features_{model_name}_{outcome}.csv"
                top_df.to_csv(top_csv, index=False)
                print(f"Saved top-features CSV -> {top_csv}")
            except Exception as e:
                print(f"[WARN] SHAP failed for {model_name} - {outcome}: {e}")


# ROC curves
for outcome in OUTCOMES:
    if outcome not in df.columns:
        continue
    y = df[outcome]
    if y.notna().sum() < 100 or y.nunique() < 2:
        continue

    X = df[feature_pool].copy()
    y = y.astype(int)
    aligner = AlignTwoCols(cols=("CD34_num", "TNC_num"), q_lo=0.01, q_hi=0.99)
    X = aligner.fit_transform(X)

    for model_name in models_to_run:
        model_path = OUTDIR / "models" / f"{model_name}_{outcome}.joblib"
        if not model_path.exists():
            continue
        pipe = joblib.load(model_path)
        out = OUTDIR / "figs" / f"roc_{model_name}_{outcome}.png"
        try:
            plot_roc(pipe, X, y, f"ROC: {model_name.upper()} - {outcome}", out)
            print(f"Saved ROC -> {out}")
        except Exception as e:
            print(f"[WARN] ROC plotting failed for {model_name} - {outcome}: {e}")


# Run metadata
meta = {
    "dataset": str(BEST_SYN),
    "rows": int(df.shape[0]),
    "random_state": RANDOM_STATE,
    "saved_models": [
        str(OUTDIR / "models" / f"{m}_{o}.joblib")
        for o in OUTCOMES for m in models_to_run
        if (OUTDIR / "models" / f"{m}_{o}.joblib").exists()
    ]
}
with open(OUTDIR / "run_info.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved run info -> {OUTDIR / 'run_info.json'}")
print("End of script.")
